[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/13_ExogenousNeuralForecasting.ipynb#copy=true)

# Exogenous Neural Forecasting and Module Synthesis

**Module 0 · Lesson 13 of 13 · Student edition**  
**Estimated class time:** 90–110 minutes  
**Source sequence:** Original Day 4  

**Prerequisite:** Lessons 9–12  

## Learning objectives

By the end of this lesson, you should be able to:

- Organize PyTorch training with Dataset, DataLoader, and LightningModule.
- Train an MLP with lagged and exogenous features.
- Synthesize how different forecasting models represent memory.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

url = 'https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike-sharing-daily.csv'
df = pd.read_csv(url, parse_dates=['dteday'])
df = df[['dteday', 'cnt', 'temp', 'hum', 'windspeed', 'workingday', 'holiday']].copy()
df.columns = ['Date', 'Rentals', 'Temp', 'Humidity', 'Windspeed', 'WorkingDay', 'Holiday']
df['Rentals_Diff'] = df['Rentals'].diff()


%pip install -q pytorch-lightning

from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as L
torch.manual_seed(42)
L.seed_everything(42)

def make_lag_matrix(series, n_lags):
    series = np.asarray(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags:t])
        y.append(series[t])
    return np.asarray(X), np.asarray(y)

def make_lag_matrix_exog(series, exog, n_lags):
    series, exog = np.asarray(series), np.asarray(exog)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(np.append(series[t - n_lags:t], exog[t]))
        y.append(series[t])
    return np.asarray(X), np.asarray(y)

def compute_metrics(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.4f}   MAE={mae:.4f}')
    return {'RMSE': rmse, 'MAE': mae}

N_LAGS = 7
TRAIN_FRAC = 0.80
target_series = df['Rentals_Diff'].dropna().values
temp_aligned = df['Temp'].iloc[1:].values

X_uni, y_uni = make_lag_matrix(target_series, N_LAGS)
split = int(len(X_uni) * TRAIN_FRAC)
X_uni_train, X_uni_test = X_uni[:split], X_uni[split:]
y_uni_train, y_uni_test = y_uni[:split], y_uni[split:]
lr_uni = LinearRegression().fit(X_uni_train, y_uni_train)
lr_uni_preds = lr_uni.predict(X_uni_test)

X_exog, y_exog = make_lag_matrix_exog(target_series, temp_aligned, N_LAGS)
split_exog = int(len(X_exog) * TRAIN_FRAC)
X_exog_train, X_exog_test = X_exog[:split_exog], X_exog[split_exog:]
y_exog_train, y_exog_test = y_exog[:split_exog], y_exog[split_exog:]
lr_exog = LinearRegression().fit(X_exog_train, y_exog_train)
lr_exog_preds = lr_exog.predict(X_exog_test)

results = {
    'Univariate (lags only)': compute_metrics(
        y_uni_test, lr_uni_preds, 'Univariate Linear'
    ),
    'Exogenous (lags + Temp)': compute_metrics(
        y_exog_test, lr_exog_preds, 'Exogenous Linear'
    ),
}


## Part 4.5 - Classes

In [ ]:
from pandas.tseries.offsets import Second
class Time:
  """
  represents the time of day.
  attributes: hour, minute, second
  """
  # the init method (initialization) is a special method that is called when an object
  # is instantiated
  def __init__(self, hour=0, minute=0, second=0):
    #assign self.hour as hour
    self.hour = hour
    self.minute = minute
    self.second = second

  def print_time(self):
    print('%.2d:%.2d:%.2d' % (self.hour, self.minute, self.second))


In [ ]:
start = Time(1,42,11)

In [25]:
start.print_time()

01:42:11


In [20]:
start.hour

1

***
## Part 5 — The MLP, Revisited With Exogenous Features

Day 3 built an MLP using only lagged values of the target, with a hand-written
training loop (`zero_grad` → forward → loss → `backward` → `step`, repeated for
every epoch). That loop pattern is correct, but you'll write it again and again as
you build more models — and it's easy to introduce a subtle bug (forgetting
`zero_grad()`, forgetting `.eval()` at the right moment) when you're rewriting it
by hand each time.

**Today we introduce PyTorch Lightning** — a thin organizational layer on top of
the exact same PyTorch you already know. Lightning doesn't change *what* your model
computes; it changes *how the training loop gets organized and run*. You already
know what a tensor is and how `nn.Module` works — Lightning builds directly on top
of both.


### 5.1 Scale the exogenous-aware data

**Your turn!** Same scaling rule as Day 3: fit on training data only.


In [30]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# FILL IN: fit_transform on train, transform on test
X_exog_train_s = scaler_X.fit_transform(X_exog_train)
X_exog_test_s  = scaler_X.transform(X_exog_test)

y_exog_train_s = scaler_y.fit_transform(y_exog_train.reshape(-1, 1)).ravel()

X_exog_train_t = torch.tensor(X_exog_train_s, dtype=torch.float32)
y_exog_train_t = torch.tensor(y_exog_train_s, dtype=torch.float32)
X_exog_test_t  = torch.tensor(X_exog_test_s,  dtype=torch.float32)

print('X_exog_train_t shape:', X_exog_train_t.shape)


X_exog_train_t shape: torch.Size([578, 8])


### 5.2 Wrap your tensors in a `Dataset` and `DataLoader`

Lightning expects training data to come from a PyTorch `DataLoader`, which serves
up your data in batches. To build one, you first need a `Dataset` — a small class
that just knows two things: how many examples you have, and how to return the
$i$-th example.

**Your turn!** Complete the `Dataset` below.


In [33]:
class LagDataset(Dataset):
    """
    A minimal PyTorch Dataset wrapping a lag-matrix (X, y) pair.
    """
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        # FILL IN: how many examples are in this dataset?
        return len(self.X)

    def __getitem__(self, idx):
        # FILL IN: return the idx-th feature row and the idx-th target
        return self.X[idx], self.y[idx]


train_dataset = LagDataset(X_exog_train_t, y_exog_train_t)

# FILL IN: wrap train_dataset in a DataLoader. Use batch_size=16, shuffle=True
# (shuffling ROWS within a single training pass is fine — this is not the same as
#  shuffling the train/test SPLIT, which we never do)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f'Dataset size: {len(train_dataset)}')
print(f'Number of batches per epoch: {len(train_loader)}')


Dataset size: 578
Number of batches per epoch: 73


> 💡 **Why shuffle rows within an epoch is fine, but shuffling the train/test split
> is not:** `shuffle=True` here only controls the *order* in which already-assigned
> training rows are fed to the model during one pass — every row was already
> determined to belong to the training set by your chronological split back in
> Part 4. No test-set information enters training either way.


### 5.3 Define the model as a `LightningModule`

A `LightningModule` is a regular `nn.Module` with three extra pieces:

- **`forward`** — exactly the same as before: takes input, returns a prediction.
- **`training_step`** — replaces your old hand-written loop body. Given one batch,
  it computes and returns the loss. Lightning handles `zero_grad()`, `backward()`,
  and `step()` for you, automatically, every batch.
- **`configure_optimizers`** — returns the optimizer (same `torch.optim.Adam` you
  already know — nothing new here).

**Your turn!** Complete the class below. The network architecture itself
(`self.net = nn.Sequential(...)`) is identical to Day 3's MLP — only the training
machinery around it changes.


In [ ]:
class LitMLPForecaster(L.LightningModule):
    def __init__(self, input_size, hidden_size=32, lr=1e-3):
        super().__init__()
        self.lr = lr
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        # FILL IN: unpack the batch into x and y
        x, y = batch
        # FILL IN: get predictions by calling self on x (same as Day 3's forward pass)
        preds = self(x)
        # FILL IN: compute MSE loss between preds and y
        loss = nn.functional.mse_loss(preds, y)
        self.log('train_loss', loss)   # Lightning logs this automatically each step
        return loss

    def configure_optimizers(self):
        # FILL IN: return an Adam optimizer over self.parameters(), using self.lr
        return torch.optim.Adam(self.parameters(), lr=self.lr)


### 5.4 Train with a `Trainer`

This is the biggest visible change from Day 3: instead of writing a `for epoch in
range(...)` loop yourself, you hand your model and your `DataLoader` to a
`L.Trainer`, and call `.fit()`. The Trainer runs the same forward → loss →
backward → step sequence you wrote by hand on Day 3 — it's just running it for you
now, the same way every time, with far less code.

**Your turn!** Set `max_epochs` and call `.fit()`.


In [35]:
# input_size must be N_LAGS + 1 (lags plus the exogenous column) — same rule as before
mlp_exog = LitMLPForecaster(input_size=N_LAGS + 1, hidden_size=32)

# FILL IN: set max_epochs to something reasonable (Day 3 used 300 manual epochs)
trainer = L.Trainer(max_epochs=100, enable_progress_bar=True, logger=False)

# FILL IN: fit the model using the train_loader you built above
trainer.fit(mlp_exog, train_dataloaders=train_loader)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net  │ Sequential │    833 │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 833                                                                                              
Non-trainable params: 0                                                                                            
Total params: 833                                                                                                  
Total estimated model params size (MB): 0.003                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


### 5.5 Generate predictions

Prediction works exactly as it did with raw PyTorch: call `.eval()`, wrap in
`torch.no_grad()`, and call the model on your input tensor. Lightning doesn't
change this part at all — `LitMLPForecaster` is still a regular `nn.Module`
underneath everything else.


In [36]:
mlp_exog.eval()
with torch.no_grad():
    mlp_exog_preds_s = mlp_exog(X_exog_test_t).numpy()

# FILL IN: inverse-transform to the original scale
mlp_exog_preds = scaler_y.inverse_transform(mlp_exog_preds_s.reshape(-1,1)).ravel()

results['MLP (lags + Temp)'] = compute_metrics(y_exog_test, mlp_exog_preds, 'MLP Exogenous')


MLP Exogenous                   RMSE=1256.2359   MAE=873.1489


### ✏️ Written Response 5

1. Did the MLP improve on the exogenous-aware linear model? On the univariate model?
2. Identify the three Lightning-specific pieces you added to `LitMLPForecaster`
   (`training_step`, `configure_optimizers`, and the `Dataset`/`DataLoader` setup).
   For each one, name the equivalent line(s) of raw-PyTorch code from Day 3's
   hand-written training loop that it replaces.
3. The MLP can learn *non-linear* combinations of lags and the exogenous variable
   (e.g., "temperature only matters when it's also a working day"). Propose one such
   interaction you think might genuinely exist in this data.
4. Would you expect adding *more* exogenous variables (humidity, windspeed, holiday)
   to keep helping indefinitely, or do you expect diminishing returns? Why?

> **YOUR ANSWER:**


***
## Part 6 — Full Comparison: Univariate vs. Exogenous-Aware

**Your turn!** Assemble everything from today into one sorted comparison table.


In [ ]:
# FILL IN: build a DataFrame from results and sort by RMSE
comparison_df = pd.DataFrame(???).T
comparison_df = comparison_df.sort_values(???)

print('=== Day 4 Model Comparison (sorted by RMSE) ===')
print(comparison_df.to_string(float_format='{:.4f}'.format))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
test_range = np.arange(len(y_exog_test))

ax.plot(test_range, y_exog_test,      label='Actual',                 color='black', linewidth=2)
ax.plot(test_range, lr_uni_preds[-len(y_exog_test):], label='Univariate Linear', linestyle='--')
ax.plot(test_range, lr_exog_preds,    label='Exogenous Linear',       linestyle='-.')
ax.plot(test_range, mlp_exog_preds,   label='MLP (Exogenous)',        color='purple')

ax.set_title('Univariate vs. Exogenous-Aware Forecasts — Test Set')
ax.set_xlabel('Test Step')
ax.legend()
plt.tight_layout()
plt.show()


### ✏️ Written Response 6

1. Rank all models tested today by RMSE. Where does adding the exogenous variable
   help most — the linear model, the MLP, or both equally?
2. If you had to deploy **one** of today's models in a real bike-share system that
   receives a weather forecast each morning, which would you choose, and why?
3. What would you need to verify before trusting this exogenous variable in
   production? *(Hint: think about Part 4's assumption that `Temp` is "known in
   advance." Is that actually true for a forecast, or only for historical data?)*

> **YOUR ANSWER:**


***
## Part 7 — Capstone Reflection: Memory Across the Week

This is a **written-only** section — no new code. Over four days, every model you've
built has answered the same underlying question in a different way: *how much does the
past (and now, the outside world) influence the present?*

| Day | Model | How it represents "memory" |
|---|---|---|
| 2 | AR($p$) | A fixed window of exactly $p$ past values, combined linearly |
| 3 | Random Forest / Gradient Boosting | The same fixed lag window, but combined through non-linear splits |
| 3 | MLP | The same fixed lag window, combined through learned non-linear functions — but lags are treated as an *unordered* set |
| 3 | RNN | A *learned, recursively updated* hidden state — order matters, but long-range influence can vanish |
| 3 | LSTM | A *gated* cell state that can preserve information across many more steps than a plain RNN |
| 4 | Exogenous models | Memory of the series' own past, **plus** same-time information from outside the series entirely |

### ✏️ Final Written Reflection

Write a **6–8 sentence capstone summary**, as if explaining the whole week to someone
who only has tonight to catch up before tomorrow. Your summary must address:

- How the notion of "memory" evolves from AR's fixed window to the LSTM's gated cell
  state — what specifically changes at each step?
- Why a lag-embedded feature matrix was the key idea that made all of this possible
  in the first place
- What an exogenous variable adds that no amount of the series' own lagged history can
  provide
- One concrete recommendation: for a *new* forecasting problem you might encounter,
  what is the first model you'd try, and what would make you reach for something more
  complex?

> **YOUR ANSWER:**


***
## Moving Forward

Choose **one more** exogenous variable from today's dataset (`Humidity`, `Windspeed`,
`WorkingDay`, or `Holiday`) and repeat Parts 4–5 using it instead of `Temp`. Add your
new model's results to the comparison table. Write 2–3 sentences: did this variable
help more or less than `Temp`? Does that match the correlation table from Part 1.3?

```python
# Your code here
```


## References / Further Reading

* [UCI Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)
* [Forecasting: Principles and Practice — Chapter 7 (Regression with ARIMA errors)](https://otexts.com/fpp3/regarima.html)
* [statsmodels SARIMAX documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html)
